# FORESEE Models: Dark Higgs

## Load Libraries 

In [ ]:
import sys, os
src_path = "../../"
sys.path.append(src_path)
import numpy as np
from src.foresee import Foresee, Utility, Model
from matplotlib import pyplot as plt

## 1. Specifying the Model

The phenomenology of the dark Higgs $\phi$ can be described by the following Lagrangian

\begin{equation}
 \mathcal{L} =  - \color{red}{m_{\phi}}^2\ \phi^2  - \sin\color{red}{\theta} \ \sum \ (m \ /\ v )\ \bar f \ f  \ \phi - \lambda \ v \ h \phi \phi
\end{equation}

with the dark Higgs mass $m_{\phi}$ and the mixing angle $\theta$ as free parameters. Additionally, one can consider an the tri-linear coupling $\lambda$ as a third parameter of the theory. In the followig, we will keep fixed at $\lambda=0.0033$ corresponding to BR$(h \to \phi\phi)=5\%$.  References [1710.09387](https://arxiv.org/pdf/1710.09387.pdf) and [1811.12522](https://arxiv.org/pdf/1811.12522.pdf) are used for calculation of branching fraction to Dark Higgs as well as decay and lifetime of Dark Higgs boson.

In [ ]:
energy = "14"
modelname = "DarkHiggs"
model = Model(modelname)

# Builder parameters, matching the build.py / load_model() defaults.
nsample_2body = 2000
nsample_3body = 2000
generators_heavy = ["NLO-P8", "NLO-P8-Max", "NLO-P8-Min"][:1]

**Production** The Dark Higgs is mainly produced in the flavour changing 2-body decay of $B$-mesons $B \to X_s \phi$. This process includes all b-flavoured hadrons and all strange-flavored decay products. 

\begin{equation}
    \text{BR}(b \to X_s\  \phi) = 5.6 \times (1-m_\phi^2/m_B^2)^2 \times \theta^2
\end{equation}

In the following, we model heavy hadron production using the `POWHEG+Pythia8` predicions.

In [ ]:
model.add_production_2bodydecay(
    pid0 = "511",
    pid1 = "130",
    br = "5.6 * coupling**2 * pow(1.-pow(mass/self.masses('pid0'),2),2)",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body,
)

model.add_production_2bodydecay(
    pid0 = "-511",
    pid1 = "130",
    br = "5.6 * coupling**2 * pow(1.-pow(mass/self.masses('pid0'),2),2)",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body,
)

model.add_production_2bodydecay(
    pid0 = "521",
    pid1 = "321",
    br = "5.6 * coupling**2 * pow(1.-pow(mass/self.masses('pid0'),2),2)",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body,
)

model.add_production_2bodydecay(
    pid0 = "-521",
    pid1 = "321",
    br = "5.6 * coupling**2 * pow(1.-pow(mass/self.masses('pid0'),2),2)",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body,
)

If the tri-linear coupling does not vanish, the dark Higgs can also be produced in 2-body decays of the Higgs boson: $h \to \phi \phi$. The corresponding branching fraction (for our choice of $\lambda=0.0033$) is

\begin{equation}
    \text{BR}(h \to \phi \  \phi) = 4700 \lambda^2 = 0.05
\end{equation}

Here the line `pid1="0"` means that the second decay product is also the LLP of interest, and `scaling=0` indicates that the branching fraction is constant wrt. the coupling $\theta$. Since we have two $\phi$ in the final state, we include an additional factor 2 in the branching fraction.  

In [ ]:
#model.add_production_2bodydecay(
#    pid0 = "25",
#    pid1 = "0",
#    br = "2*0.05",
#    generator = ["Pythia8"],
#    energy = energy,
#    nsample = 200,
#    scaling = 0,
#)

The dark Higgs can also be produced in 3-body decays $B \to X_s \phi \phi$ via an offshell Higgs boson. This can be added using the function `add_production_3bodydecay()`. It requires to provide `br` which is the differential branching fraction $d\text{BR}/(dq^2 \ d\cos\vartheta)$ where $q^2=(p_{\phi_1}+p_{\phi_2})^2$ and $\vartheta$ is the angle between $p_{\phi_1}$ in the restframe of $p_{\phi_1}+p_{\phi_2}$ and the direction of $p_{\phi_1}+p_{\phi_2}$ in the restframe of the $b$. Since the process is mediated by a an offshell scalar (the Higgs), there is no dependence on the angle $\cos\vartheta$. We can write for the branching ratio

\begin{equation}
    \frac{d\text{BR} (\phi)}{d q^2 d\cos\vartheta} 
    = \frac{(4.9\cdot 10^{-8} {\rm GeV}^{-2} \ \lambda)^2 m_b^3}{512 \ \pi^3 \  \Gamma_b} 
      \times \left(1-\frac{4 m_\phi^2}{q^2}\right)^{1/2} \!\!\!\!\! \times \left(1-\frac{q^2}{m_b^2}\right)^2
    = 3.68\cdot 10^{-10}  \times \left(1-\frac{4 m_\phi^2}{q^2}\right)^{1/2}  \!\!\!\!\! \times \left(1-\frac{q^2}{m_b^2}\right)^2
\end{equation}

Since we have two $\phi$ in the final state, we again multiply the BR by an additional factor 2. 
Furthermore, as the factor $3.68\cdot 10^{-10}$ assumed a hard-coded value of $m_b^3=4.5^3$ GeV$^3$, a factor of $(m_b/4.5)^3$ ensures consistency with the $m_b$ definition in `src/foresee.py`.

In [ ]:
model.add_production_3bodydecay(
    label= "511_di",
    pid0 = "511",
    pid1 = "130",
    pid2 = "0",
    br = "7.37e-10*(self.masses('5')/4.5)**3*np.sqrt(1-4*mass**2/q**2)*(1-q**2/self.masses('5')**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_3body,
    scaling = 0, 
)

model.add_production_3bodydecay(
    label= "-511_di",
    pid0 = "-511",
    pid1 = "130",
    pid2 = "0",
    br = "7.37e-10*(self.masses('5')/4.5)**3*np.sqrt(1-4*mass**2/q**2)*(1-q**2/self.masses('5')**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_3body,
    scaling = 0, 
)

model.add_production_3bodydecay(
    label= "521_di",
    pid0 = "521",
    pid1 = "321",
    pid2 = "0",
    br = "7.37e-10*(self.masses('5')/4.5)**3*np.sqrt(1-4*mass**2/q**2)*(1-q**2/self.masses('5')**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_3body,
    scaling = 0, 
)

model.add_production_3bodydecay(
    label= "-521_di",
    pid0 = "-521",
    pid1 = "321",
    pid2 = "0",
    br = "7.37e-10*(self.masses('5')/4.5)**3*np.sqrt(1-4*mass**2/q**2)*(1-q**2/self.masses('5')**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_3body,
    scaling = 0, 
)

**Decay:** Dark Higgs bosons can decay into all kinematically accessible light charged states.

In [ ]:
model.set_ctau_1d(
    filename="model/ctau.txt", 
    coupling_ref=1
)

decay_modes = ["e_e", "mu_mu", "K_K", "pi_pi"]
model.set_br_1d(
    modes=decay_modes,
    finalstates=[[11,-11], [13,-13], [321,-321], [211,-211]],
    filenames=["model/br/"+mode+".txt" for mode in decay_modes],
)

We can now initiate FORESEE with the model that we just created. 

In [ ]:
foresee = Foresee(path=src_path)
foresee.set_model(model=model)

## 2. Event Generation

In the following, we want to study one specific benchmark point with $m_{\phi}=1.5$ GeV and $\theta=10^{-4}$ and export events as a HEPMC file. 

In [ ]:
mass, coupling, = 1.5, 1e-4

First, we will produce the corresponding flux for this mass and a reference coupling $\theta_{ref}=1$.

In [ ]:
plot=foresee.get_llp_spectrum(mass=mass, coupling=1, do_plot=True)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Spectrum_{modelname}.pdf", bbox_inches="tight")
plot.show()

Next, let us define the configuration of the detector (in terms of position, size and luminosity). Here we choose FASER2 at the FPF. 

In [ ]:
foresee.set_detector(
    distance=620, 
    selection="np.sqrt(x.x**2 + x.y**2)<1", 
    length=10, 
    luminosity=3000, 
)

For our benchmark point, let us now look at how many particle decay inside the decay volume. We also export 1000 unweighted events as a HEPMC file. 

In [ ]:
setupnames = ['POWHEG-central', 'POWHEG-max', 'POWHEG-min'][:1]

momenta, weights, _ = foresee.write_events(
    mass = mass, 
    coupling = coupling, 
    energy = energy, 
    numberevent = 1000,
    filename = "model/events/test.hepmc", 
    return_data = True,
    weightnames=setupnames,
    modes=None,
)

for isetup, setup in enumerate(setupnames):
    print("Expected number of events for "+setup+":", round(sum(weights[:,isetup]),3))

Let us plot the resulting energy distribution

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = plt.subplot(1,1,1)
energies = [p.e for p in momenta], 
for isetup, setup in enumerate(setupnames):
    ax.hist(energies, weights=weights[:,isetup], bins=np.logspace(2,4, 20+1), histtype='step', label=setup) 
ax.set_xscale("log")
ax.set_xlim(1e2,1e4) 
ax.set_xlabel("E [GeV]") 
ax.set_ylabel("Number of Events per Bin") 
ax.legend(frameon=False, labelspacing=0, fontsize=14, loc='upper left')
os.makedirs(f"figures/{modelname}", exist_ok=True)
plt.savefig(f"figures/{modelname}/E_distribution_{modelname}.pdf", bbox_inches="tight")
plt.show()

## 3. Sensitivity Reach

In the following, we will obtain the projected sensitivity for the LLP model. For this, we first define a grid of couplings and masses, and then produce the corresponding fluxes. 

In [ ]:
masses=[round(x,5) for x in np.logspace(-1,np.log10(6.0),50)]
# Extra points around each production channel kinematic endpoint, from
# utility.production_thresholds(model, mass_range=(0.01, 2.05)).
thresholds = [
    2.3202, 2.39196, 2.46372, 4.6404, 4.78392, 4.92744,
]
masses = sorted(masses + thresholds)
couplings = np.logspace(-6,-2,100) 

# Use cached LLP spectra: get_llp_spectrum recomputes on every call,
# so skip any masses already saved in model/LLP_spectra/.
for mass in masses:
    if not os.path.exists(f"model/LLP_spectra/{energy}TeV_m_{mass}.txt.gz"):
        foresee.get_llp_spectrum(mass=mass, coupling=1)

We can now plot the `production rate vs mass` using the `foresee.plot_production()` function. Below the production rates, we also show the decay branching fractions of the leading visible final states.

In [ ]:
productions=[
     {"channels": ["511","-511","521","-521"]             , "color": "red"  , "label": r"$B \to X_s \phi$"      , "generators": generators_heavy},
     {"channels": ["511_di","-511_di","521_di","-521_di"] , "color": "blue" , "label": r"$B \to X_s \phi\phi$"  , "generators": generators_heavy},
]

branchings = [
    ["e_e"       , "red"        , "solid" , r"$e^+e^-$"         , 0.110, 0.50],
    ["mu_mu"     , "magenta"    , "solid" , r"$\mu^+\mu^-$"     , 0.350, 0.2],
    ["pi_pi"     , "blue"       , "dashed", r"$\pi^+\pi^-$"     , 0.350, 0.50],
    ["K_K"       , "forestgreen", "dashed", r"$K^+K^-$"         , 0.700, 0.09]
]

plot, ax, ax2 = foresee.plot_production(
    masses = masses,
    productions = productions,
    energy=energy,
    condition="logth<-3.7 and logp>2",  
    xlims=[0.1,10],ylims=[1e-7,4e7],
    xlabel=r"Mass [GeV]",
    ylabel=r"Production Rate $\sigma/\theta^2$ [pb]",
    title=r"$\theta < 0.2$ mrad and $E > 100$ GeV",
    legendloc=(0.97,1),
    fs_label=12,
    ncol=2,
    figsize=(7,6),
    fs_label_br=9,
    branchings=branchings,
)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Production_{modelname}.pdf", bbox_inches="tight")


Let us now scan over various masses and couplings, and record the resulting number of evets. Note that here we again consider the FASER2 configuration, which we set up before.

In [ ]:
setupnames = ['POWHEG-central']
modes = None

if energy == "13.6": detectors = [["FASER_R3"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 250 ,  None]]
elif energy == "14": detectors = [["FASER_HL"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 3000,  None], 
                                  ["FASER2_HL" , 650, "-1.5<x.x<1.5 and -.5<x.y<.5" , 10 , 3000,  None]]

condition = f"np.sqrt(p**2 + mass**2) > 100"

for detector in detectors: 

    #setup detector
    dlabel, distance, selection, length, luminosity, channels  = detector

    #skip detectors already precomputed (the plot cell reads them); scan only missing ones
    if all(os.path.exists(f"model/results/{energy}TeV_{dlabel}_{label}.npy") for label in setupnames):
        continue

    foresee.set_detector(distance=distance, selection=selection, length=length, luminosity=luminosity, channels=channels)

    #get reach  
    list_nevents = {label:[] for label in setupnames}
    for mass in masses:
        couplings, _, nevents, _, _  = foresee.get_events(mass=mass, energy=energy, couplings = couplings,modes=modes,nsample=10, preselectioncuts = condition)
        for i,label in enumerate(setupnames): list_nevents[label].append(nevents.T[i])  
            
    #save results
    configuration=dlabel
    for label in setupnames: 
        result = np.array([masses,couplings,list_nevents[label]], dtype='object')
        np.save("model/results/"+energy+"TeV_"+configuration+"_"+label+".npy",result)

We can now plot the results. For this, we first specify all detector setups for which we want to show result (filename in model/results directory, label, color, linestyle, opacity alpha for filled contours, required number of events).

In [ ]:
setups = [ 
    ["13.6TeV_FASER_R3_POWHEG-central.npy"   , r"FASER (Run 3)"    , "firebrick"         ,  "solid"  , 0., 3],
    ["14TeV_FASER_HL_POWHEG-central.npy"   , r"FASER (HL-LHC)"    , "red"         ,  "dashed"  , 0., 3],
    ["14TeV_FASER2_HL_POWHEG-central.npy"   , r"FASER2 (HL-LHC)"    , "salmon"         ,  "dashed"  , 0., 3],    
]

Then we specify all the existing bounds (filename in model/bounds directory, label, label position x, label position y, label rotation)

In [ ]:
bounds = [ 
    ["bounds_1508.04094.txt", r"LHCb $B^0$"  , 0.430, 0.001,  90 ],
    ["bounds_1612.08718.txt", r"LHCb $B^+$"  , 0.330, 0.001,  90 ],
    ["bounds_1612.08718.txt", r"LHCb $B^+$"  , 2.500, 0.001,  90 ],
    ["bounds_LSND.txt"      , "LSND"        , 0.250, 4.9e-5, 90 ],
    ["bounds_CHARM.txt"     , "CHARM"       , 0.250, 1.8e-4, 90 ],
    ["bounds_MicroBoone.txt", r"$\mu$BooNE"  , 0.138, 1.2e-4, 90 ],
    ["bounds_E949.txt"      , "E949"        , 0.102, 8.6e-5, 90 ],
    ["bounds_2011.11329.txt", r"NA62 $K^+$"  , 0.170, 2.8e-4, 90 ],
    ["bounds_2010.07644.txt", r"NA62 $\pi^+$", 0.125, 1.1e-3, 90 ],
]

We then specify other projected sensitivitities (filename in model/bounds directory, color, label, label position x, label position y, label rotation)

In [ ]:
projections = [
    # ["limits_SHiP.txt",       "teal",         "SHiP"    , 2.700, 3.2*10**-5, 0  ],
    # ["limits_MATHUSLA.txt",   "dodgerblue",   "MATHUSLA", 0.120, 5.0*10**-6, 0  ],
    # ["limits_CodexB.txt",     "deepskyblue",  "CodexB"  , 1.700, 2.0*10**-5, 0  ],
    # ["limits_LHCb.txt",       "cyan",         "LHCb"    , 3.800, 1.0*10**-4, 0  ],
]

Finally, we can plot everything using `foresee.plot_reach()`.

In [ ]:
plot = foresee.plot_reach(
    setups=setups,
    bounds=bounds,
    projections=projections,
    title="Dark Higgs", 
    xlims=[0.1,10], 
    ylims=[8e-7,4e-3],
    xlabel=r"Dark Higgs Mass $m_{\phi}$ [GeV]", 
    ylabel=r"Mixing $\theta$",
    legendloc=(1,0.22),
    linewidths=2,
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Reach_{modelname}.pdf", bbox_inches="tight")
plot.show()